In [51]:
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import gradio as gr

In [52]:
load_dotenv(override=True)
openai = OpenAI()

In [53]:
def greet(name):
    return f"hello {name}!"

demo = gr.Interface(fn=greet,inputs="text",outputs="text")

demo.launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


In [54]:
reader = PdfReader("me/Naveen_Aila_Linkedin.pdf")

print(reader.pages)



[PageObject(0), PageObject(1), PageObject(2), PageObject(3)]


In [55]:
print(reader.pages[0].extract_text())

   
Contact
+14193786358 (Mobile)
naveen.aila77@gmail.com
www.linkedin.com/in/naveenaila
(LinkedIn)
Top Skills
Data Engineering
Amazon Web Services (AWS)
Business Intelligence (BI)
Languages
English (Full Professional)
Telugu (Native or Bilingual)
Hindi (Full Professional)
Certifications
Big Data, University of California,
San Diego
Dale Carnegie Course
Tableau 9 Advanced Training
Honors-Awards
FedEx Solutions Purple Choice
Award FY19Q1
FY19Q3 FedEx Trade Networks
GSO Sales Warrior Award 
FY19Q4 FedEx Trade Networks
GSO Sales Warrior Award 
FY21Q2
Naveen Aila
BI-Analytics Leader @Amazon | MS in Data Science
Canada
Summary
Amazon’s transportation team leverages the expertise of a Business
Intelligence to deliver impactful data-driven insights. With over three
years in the BIE II role, I have demonstrated advanced proficiency in
SQL, Python and scalable analytics solutions that enable operational
excellence. Recent projects focus on optimizing transportation
execution through interactive

In [56]:
Linkedin_text = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        Linkedin_text += text


In [57]:
print(Linkedin_text)

   
Contact
+14193786358 (Mobile)
naveen.aila77@gmail.com
www.linkedin.com/in/naveenaila
(LinkedIn)
Top Skills
Data Engineering
Amazon Web Services (AWS)
Business Intelligence (BI)
Languages
English (Full Professional)
Telugu (Native or Bilingual)
Hindi (Full Professional)
Certifications
Big Data, University of California,
San Diego
Dale Carnegie Course
Tableau 9 Advanced Training
Honors-Awards
FedEx Solutions Purple Choice
Award FY19Q1
FY19Q3 FedEx Trade Networks
GSO Sales Warrior Award 
FY19Q4 FedEx Trade Networks
GSO Sales Warrior Award 
FY21Q2
Naveen Aila
BI-Analytics Leader @Amazon | MS in Data Science
Canada
Summary
Amazon’s transportation team leverages the expertise of a Business
Intelligence to deliver impactful data-driven insights. With over three
years in the BIE II role, I have demonstrated advanced proficiency in
SQL, Python and scalable analytics solutions that enable operational
excellence. Recent projects focus on optimizing transportation
execution through interactive

In [58]:
with open("me/summary_Naveen.txt", "r",encoding="utf-8") as f:
    summary = f.read()


In [59]:
name = 'Naveen Aila'

In [60]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{Linkedin_text}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [61]:
system_prompt

"You are acting as Naveen Aila. You are answering questions on Naveen Aila's website, particularly questions related to Naveen Aila's career, background, skills and experience. Your responsibility is to represent Naveen Aila for interactions on the website as faithfully as possible. You are given a summary of Naveen Aila's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Naveen Aila. I'm an, software engineer and data engineer. I'm originally from Jangaon, Telangana India, but I moved to USA in 2015. Moved to Vancouver, Canada in 2023 and currently living here. \nI love all foods, particularly love Indian food, Sushi, Momo's and Italian food. I do Watch lot of flood and travel vlogs. Love travelling and doing Photography especially landscape and wildlife. Love playing guitar and listening

In [62]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [63]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [73]:
# Create a Pydantic model for the Evaluation

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [74]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{Linkedin_text}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [75]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [76]:
import os
gemini = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"), 
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [77]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = gemini.beta.chat.completions.parse(model="gemini-2.5-flash", messages=messages, response_format=Evaluation)
    return response.choices[0].message.parsed

In [ ]:
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
reply = response.choices[0].message.content
reply

"As of now, I do not hold any patents. My focus has primarily been on software engineering, data engineering, and business intelligence within my professional career. If you have any specific questions about my projects or contributions in those areas, I'd be happy to share!"

In [78]:
def rerun(reply,message,history,feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

In [79]:
def chat(message,history):
    if "patent" in message:
        system = system_prompt + "\n\n everything in your reply needs to be in pig latin -\
             it is mandatory that you respond only and entirely in pig latin " 
    else:
        system = system_prompt
    
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4o-mini", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply,message,history)

    if evaluation.is_acceptable:
         print("Passed evaluation - returning reply")
    else:
         print("Did not Pass evaluation - rerunning")
         reply = rerun(reply,message,history,evaluation.feedback)

    return reply

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7873
* To create a public link, set `share=True` in `launch()`.


Passed evaluation - returning reply
Did not Pass evaluation - rerunning
